# 4.4 A Phenome-Wide Association Study (PheWAS)

**Omics series, class 4 of 7.** In 4.2 and 4.3 we asked: *which variants are linked to type 2 diabetes?* A **PheWAS** flips the question: *which diseases are linked to one variant?* We take **rs7903146**, the *TCF7L2* variant from 4.1 to 4.3, and test it against **hundreds of diseases** found in the health records.

We do it twice:

1. **With All by All**, the *All of Us* website of ready-made results: no code, no cost
2. **Ourselves**, in R, on a random sample of 20,000 participants, to see how a PheWAS is built

### By the end of this notebook you'll be able to

1. Explain what a PheWAS tests, and how it differs from a GWAS
2. Explain **phecodes**: how diagnosis codes are grouped into diseases, with case and control rules
3. Read PheWAS results in the **All by All** browser, including converting a beta to an odds ratio
4. Build phecode phenotypes from the EHR, and run a PheWAS with `glm()`
5. Explain why a PheWAS needs a **random** sample, and why many hits are **consequences** of one disease

| | |
|---|---|
| ⏱️ **Time** | 120 minutes |
| 📂 **Uses** | the Controlled Tier dataset (BigQuery), the v9 genomic files, and public phecode lookup tables |
| 💾 **Saves** | `phewas_rs7903146.csv`: the results |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts | 20 |
| 2. Explore All by All (no code) | 20 |
| 3. A random sample, and their genotypes | 15 |
| 4. Diagnosis codes and phecodes | 25 |
| 5. Run the PheWAS | 15 |
| 6. Read the results | 20 |
| Wrap-up | 5 |

**Before you start:**

- This notebook works **on its own**: it doesn't need 4.1 to 4.3 to have been run (but it builds on their ideas).
- Like 4.1 and 4.2, it's an **R** notebook in `notebooks/omics/`, and loads `../aou_setup.R`.

> 💰 **Cost:** runs on a **standard app**. Two BigQuery queries (the cohort, and diagnosis codes for 20,000 people): together about **$1 or less**. PLINK reads one variant in seconds. The PheWAS itself runs on the app in a few minutes.

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data (including genotypes, IDs, and diagnosis codes), or counts of 1 to 20. We only test diseases with at least 100 cases. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### GWAS vs PheWAS

| | GWAS (4.2, 4.3) | PheWAS (today) |
|---|---|---|
| Question | which **variants** are linked to this disease? | which **diseases** are linked to this variant? |
| One test per | variant | disease (phenotype) |
| Number of tests | thousands to millions | hundreds to about 1,800 |
| The fixed part | the disease: type 2 diabetes | the variant: rs7903146 |
| The model | logistic regression, adjusted for age, sex, PCs | **the same** |

A PheWAS can show that a variant affects **more than one** trait (**pleiotropy**), find **consequences** of a disease, and check that a known association **replicates** in a new dataset.

### Phecodes: from diagnosis codes to diseases

The EHR records diagnoses as **ICD codes**: ICD-9-CM (until 2015 in the US) and ICD-10-CM (since). There are about 70,000 ICD-10-CM codes, often very specific: `E11.9` is *type 2 diabetes without complications*, `E11.65` is *type 2 diabetes with hyperglycemia*. Too many, too specific, and two coding systems.

**Phecodes** group ICD codes into about **1,800 diseases** designed for research. For example, `E11.9`, `E11.65`, and ICD-9 `250.00` all map to phecode **250.2, Type 2 diabetes**.

| Rule | Means | Why |
|---|---|---|
| **Roll-up** | phecodes form a hierarchy: 250.2 (*Type 2 diabetes*) is under **250** (*Diabetes mellitus*). A person with 250.2 also counts for 250 | so broad and specific diseases are both tested |
| **Case** | at least **2 days** with a code for the phecode | one code can be a mistake, or a test to *rule out* a disease. All by All uses the same rule |
| **Excluded** | people with only **1** day, or with a **related** phecode (e.g. type 1 diabetes, when testing type 2) | they're neither clearly cases nor clearly free of the disease |
| **Control** | everyone else | |
| **Sex-specific** | some phecodes (e.g. prostate cancer) are tested in one sex only | |

The phecode lookup tables are free and public (from the **PheWAS** project at Vanderbilt). We load them from the PheWAS project's GitHub page.

### Why a random sample this time?

In 4.1 to 4.3 we used 2,500 **cases** and 2,500 **controls** for type 2 diabetes. That design is right for a GWAS of diabetes, but **wrong for a PheWAS**:

- In that sample, half the people have diabetes, and rs7903146's T allele is more common in them.
- So in that sample, **anything that goes with diabetes** (high blood pressure, kidney disease, obesity, diabetes medicines) also goes with the T allele, just because of how the sample was chosen.
- A PheWAS would then "find" links between rs7903146 and all of those, made up by the sampling. This is **selection bias**.

So today we take a **random sample of 20,000** adults with a genome, from everyone with EHR data, whatever their diabetes status.

### Direct effects, or consequences?

Even in a random sample, a variant that causes diabetes will be linked to diabetes **medicines**, high **blood glucose**, and diabetes **complications**. Those are real, but they're **consequences of diabetes**, not separate effects of the variant. Part 6 tests this by adjusting for diabetes.

### Glossary

| Term | Means |
|---|---|
| **PheWAS** | phenome-wide association study: one variant, many phenotypes |
| **Phenome** | all the phenotypes (traits and diseases) we can measure |
| **ICD-9-CM, ICD-10-CM** | the US diagnosis coding systems in the EHR |
| **Phecode** | a research grouping of ICD codes into one disease |
| **Pleiotropy** | one variant affecting several traits |
| **Selection bias** | a misleading association created by how the sample was chosen |
| **Collider bias** | a misleading association created by adjusting for a common **effect** of two variables (Part 6) |
| **Beta** | the log odds ratio; **OR = e^beta** |

## Part 2. Explore All by All (no code)

**All by All** is a set of ready-made association results from *All of Us*: thousands of phenotypes tested against the genome, by the Broad Institute and *All of Us*. Running a study like it yourself would cost a fortune; looking up the results is free.

The browser is public: **[allbyall.researchallofus.org](https://allbyall.researchallofus.org)**. Open it in a new tab (it's outside the Workbench, and has no participant-level data).

### Find rs7903146

1. In the search box, type **`chr10-112998590-C-T`** (chromosome-position-REF-ALT, the variant's standard name) and open the result.
2. The page shows **every phenotype** tested against this variant: the top plot is the PheWAS (each dot is a phenotype; higher = smaller p-value), the lower plot shows each **beta**, and the table lists them.
3. Sort the table by **P-Value**.

**🔍 What to look for:**

- The top hits are **type 2 diabetes**, **diabetes mellitus**, **blood glucose**, **HbA1c**, and diabetes **medicines** (biguanides such as metformin, insulins, sulfonylureas).
- **Convert a beta to an odds ratio:** for type 2 diabetes, beta ≈ 0.22, so OR = e^0.22 ≈ **1.24** per T allele. Compare with 4.2 and 4.3 (about 1.25 in our test run). The p-value in All by All is far smaller (about 10⁻¹⁵²): it used about 50,000 cases instead of our 2,500.
- **N cases / N controls:** the numbers behind each test. Lab values and measurements show 0 controls because they're **quantitative** (a value for everyone), not case/control.

### ✍️ Your turn (in the browser)

1. Find **waist circumference** in the table. Is its beta positive or negative? What does that mean for people with the T allele?
2. Search for the gene **`TCF7L2`** instead of the variant. The gene page shows **burden tests**: each person's **rare** variants that break the gene, tested together. Are any phenotypes significant?

<details>
<summary><b>Show answer</b></summary>

1. In our test, waist circumference had a **negative** beta (about −0.015, p ≈ 4 × 10⁻⁸): the T allele goes with a slightly **smaller** waist. It raises diabetes risk through **insulin secretion** (how the pancreas releases insulin), **not** through obesity. A PheWAS can show this kind of surprise.
2. In our test, **nothing** was significant for the rare-variant burden tests. A **common** variant with a modest effect (rs7903146) and **rare** variants that break the gene can tell very different stories: rs7903146 is thought to change **how much** of the gene is made, not to break it.

</details>

👩‍🏫 **Instructor tip:** the browser shows meta-analysis results across ancestry groups. The Controlled Tier has the full results (per ancestry group, as Hail tables) for anyone who needs more than the browser; see the *All of Us* Support Hub article on All by All.

## Part 3. A random sample, and their genotypes

### Setup

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

AOU_DIR  <- aou_data_dir("v9")
pgen_dir <- file.path(AOU_DIR, "wgs", "short_read", "snpindel", "acaf_threshold", "pgen")
GENO_DIR <- file.path(DATA_DIR, "genomics")
dir.create(GENO_DIR, showWarnings = FALSE)

LEAD_ID  <- "chr10:112998590:C"   # rs7903146, as named in the All of Us files (4.1)
LEAD_POS <- 112998590
PCS      <- paste0("PC", 1:10)

### 20,000 people, chosen at random

`genomic_people()`, a helper in `aou_setup.R`, does the steps from 4.1 Parts 3 and 4 for **everyone** in the cohort query: people with a genome, not on the relatedness list, aged 18 to 100, sex at birth *Female* or *Male*, with ancestry and PCs. Unlike 4.1, it keeps **everyone**: cases, controls, and the people our cohort query excludes (e.g. type 1 diabetes).

Then `slice_sample()` picks 20,000 at random, **without** grouping by diabetes status. 💰 This runs the cohort query (cents to under $1).

In [ ]:
N_PEOPLE <- 20000

people <- genomic_people()
nrow(people)   # everyone eligible

set.seed(2026)
phewas_samples <- people %>% slice_sample(n = N_PEOPLE)

phewas_samples %>% count(cohort) %>% mutate(percent = round(100 * n / sum(n), 1))

**🔍 Read the output:** the first number is everyone eligible (over 300,000 people). In our random 20,000, about **1 in 5** has type 2 diabetes (`case`): that's how common it is in *All of Us* participants with EHR data, not 50% as in our GWAS sample. Diabetes status didn't affect who was picked.

### rs7903146 genotypes

The same PLINK step as in 4.1, for one variant: count copies of the **T** allele (`--export-allele`) for our 20,000 people. PLINK jumps to the variant, so this takes seconds even though the file has 535,000 people.

In [ ]:
keep_file   <- file.path(GENO_DIR, "keep_phewas.txt")
allele_file <- file.path(GENO_DIR, "count_allele.txt")
lead_prefix <- file.path(GENO_DIR, "rs7903146_phewas")

write_lines(phewas_samples$person_id, keep_file)
write_lines(str_glue("{LEAD_ID}\tT"), allele_file)

plink_output <- system(str_glue(
  "plink2 --pfile {shQuote(file.path(pgen_dir, 'acaf_threshold.chr10'))} ",
  "--keep {shQuote(keep_file)} --chr 10 --from-bp {LEAD_POS} --to-bp {LEAD_POS} ",
  "--export A --export-allele {shQuote(allele_file)} --out {shQuote(lead_prefix)}"
), intern = TRUE)

genotypes <- read_tsv(paste0(lead_prefix, ".raw"), col_types = cols(.default = "c"))
geno_col  <- names(genotypes)[ncol(genotypes)]
geno_col   # should end in _T

phewas_samples <- phewas_samples %>%
  left_join(genotypes %>% transmute(person_id = IID, dosage = as.numeric(.data[[geno_col]])),
            by = "person_id") %>%
  mutate(sex_male = if_else(sex_at_birth == "Male", 1, 0))

# T allele frequency in our random sample
round(sum(phewas_samples$dosage, na.rm = TRUE) / (2 * sum(!is.na(phewas_samples$dosage))), 3)

**🔍 Read the output:** the column name ends in `_T`, and the T allele frequency is close to *All of Us* overall (`AF=0.276` in the `.pvar` file, 4.1).

## Part 4. Diagnosis codes and phecodes

### Get everyone's ICD codes

In the OMOP tables, the **original** ICD code of a diagnosis is kept in the **source** concept column (`condition_source_concept_id`); the standard column (`condition_concept_id`, used in our cohort query) holds a SNOMED concept. Phecodes are built from ICD codes, so we use the source column.

ICD codes appear in two tables, and we read both, as other PheWAS tools do:

| Table | Column with the ICD code | Date |
|---|---|---|
| `condition_occurrence` | `condition_source_concept_id` | `condition_start_date` |
| `observation` | `observation_source_concept_id` | `observation_date` |

The query joins to `concept` to turn each concept ID into its **vocabulary** (`ICD9CM` or `ICD10CM`) and **code** (e.g. `E11.9`), and counts the **number of different days** each person has each code.

We list our 20,000 IDs inside the query (`WHERE person_id IN (...)`), so BigQuery only returns their codes. **Don't print `icd_sql`**: it contains participant IDs.

💰 It reads a few columns of two large tables: well under $1. It takes a minute or two, plus time to download a few million rows.

In [ ]:
ids_sql <- paste(phewas_samples$person_id, collapse = ", ")

icd_sql <- str_glue("
WITH events AS (
  SELECT person_id, condition_source_concept_id AS concept_id, condition_start_date AS event_date
  FROM `{CURRENT_DATASET}.condition_occurrence`
  WHERE person_id IN ({ids_sql})
  UNION ALL
  SELECT person_id, observation_source_concept_id AS concept_id, observation_date AS event_date
  FROM `{CURRENT_DATASET}.observation`
  WHERE person_id IN ({ids_sql})
)
SELECT
  events.person_id,
  concept.vocabulary_id,
  concept.concept_code AS code,
  COUNT(DISTINCT events.event_date) AS n_days
FROM events
JOIN `{CURRENT_DATASET}.concept` AS concept
  ON events.concept_id = concept.concept_id
WHERE concept.vocabulary_id IN ('ICD9CM', 'ICD10CM')
GROUP BY events.person_id, concept.vocabulary_id, concept.concept_code
")

icd <- run_query(icd_sql) %>% mutate(person_id = as.character(person_id))

nrow(icd)                         # person-code pairs
n_distinct(icd$person_id)         # people with at least one ICD code
icd %>% count(vocabulary_id)      # ICD-9-CM vs ICD-10-CM

**🔍 Read the output:** a few million rows: one per person per ICD code, with `n_days`. Almost everyone has at least one code (they all have EHR data). ICD-10-CM codes outnumber ICD-9-CM: most records are recent.

### Load the phecode lookup tables

The tables come from the **PheWAS** R package's GitHub page. The package itself doesn't install in this app (one of the packages it needs fails to build), but we only need its **data**, which we download as `.rda` files (R's format for saved objects) and `load()`.

| Table | Contains |
|---|---|
| `phecode_map` | ICD code → phecode (`vocabulary_id`, `code`, `phecode`) |
| `phecode_rollup_map` | phecode → itself and its **parents** (e.g. 250.2 → 250.2 and 250) |
| `phecode_exclude` | for each phecode, the **related** phecodes that exclude someone from being a control |
| `sex_restriction` | phecodes for one sex only |
| `pheinfo` | each phecode's **description** and disease **group** |

In [ ]:
PHECODE_URL   <- "https://raw.githubusercontent.com/PheWAS/PheWAS/master/data/"
phecode_files <- c("phecode_map.rda", "phecode_rollup_map.rda", "phecode_exclude.rda",
                   "sex_restriction.RData", "pheinfo.rda")

for (f in phecode_files) {
  path <- file.path(tempdir(), f)
  download.file(paste0(PHECODE_URL, f), path, quiet = TRUE)
  load(path)   # creates a data frame with the same name as the file
}

head(phecode_map)
nrow(pheinfo)   # phecodes

pheinfo %>% filter(phecode %in% c("250", "250.1", "250.2"))

**🔍 Read the output:** each ICD code maps to a phecode; there are about 1,800 phecodes; and 250 (*Diabetes mellitus*) has children 250.1 (*Type 1*) and 250.2 (*Type 2*).

### ICD codes → phecodes, for each person

| Step | Function |
|---|---|
| map each ICD code to its phecode | `inner_join(phecode_map, ...)`. Codes with no phecode drop out; a few codes map to **more than one** phecode, hence `relationship = "many-to-many"` |
| add each phecode's parents | `inner_join(phecode_rollup_map, ...)` |
| add up the days per person and phecode | `group_by()` + `summarise(n_days = sum(n_days))` |

Adding up days across different ICD codes is an approximation: two codes for the same phecode on the **same** day count as 2 days. The PheWAS package does the same.

In [ ]:
person_phecodes <- icd %>%
  inner_join(phecode_map, by = c("vocabulary_id", "code"), relationship = "many-to-many") %>%
  inner_join(phecode_rollup_map, by = c("phecode" = "code"), relationship = "many-to-many") %>%
  group_by(person_id, phecode = phecode_unrolled) %>%
  summarise(n_days = sum(n_days), .groups = "drop")

nrow(person_phecodes)
n_distinct(person_phecodes$phecode)   # phecodes anyone in our sample has

### Cases, controls, and excluded, for one phecode

`phecode_status()` applies the rules from Part 1 to one phecode, for all 20,000 people: **1** = case, **0** = control, **NA** = excluded. Read it line by line: it's the heart of a PheWAS.

In [ ]:
phecode_status <- function(pc) {
  cases <- person_phecodes %>%
    filter(phecode == pc, n_days >= 2) %>%            # at least 2 days with a code
    pull(person_id)

  related  <- phecode_exclude %>% filter(code == pc) %>% pull(exclusion_criteria)
  excluded <- person_phecodes %>%
    filter(phecode %in% c(pc, related)) %>%           # 1 day only, or a related phecode
    pull(person_id)

  case_when(phewas_samples$person_id %in% cases    ~ 1,
            phewas_samples$person_id %in% excluded ~ NA_real_,
            TRUE                                   ~ 0)
}

# Type 2 diabetes (phecode 250.2), compared with our cohort query's definition
phewas_samples %>%
  mutate(phecode_250.2 = phecode_status("250.2")) %>%
  count(cohort, phecode_250.2) %>%
  hide_small_counts()

**🔍 Read the output:** two definitions of type 2 diabetes, side by side:

- Our **cohort query** (`cohort`): any type 2 diabetes code (via SNOMED concepts), no type 1.
- The **phecode** (`phecode_250.2`): at least 2 days with a type 2 diabetes ICD code, and no related diabetes phecode.

Most people agree (`case` with 1, `control` with 0). Many of our cases are `NA` for the phecode: they have a type 2 diabetes code on only **one** day (about 1 in 6 cases in the full cohort), or also a code for another kind of diabetes. **Different definitions give different cohorts**; always report which one you used.

## Part 5. Run the PheWAS

### Which phecodes to test

We test phecodes with at least **100 cases** in our sample. With fewer, an odds ratio is very uncertain, and the counts could be small enough to identify people. (All by All used the same minimum.)

In [ ]:
MIN_CASES <- 100

phecodes_to_test <- person_phecodes %>%
  filter(n_days >= 2) %>%
  count(phecode, name = "n_cases_max") %>%     # cases before exclusions
  filter(n_cases_max >= MIN_CASES) %>%
  pull(phecode)

length(phecodes_to_test)

### One regression per phecode

`test_phecode()` builds the phenotype, then runs the **same model as 4.2**: logistic regression of case status on the number of T alleles, adjusted for age, sex, and 10 PCs. For a sex-specific phecode, it keeps one sex and drops `sex_male` from the model.

| Function | What it does |
|---|---|
| `reformulate(c("dosage", ...), response = "status")` | builds the formula `status ~ dosage + age + ...` from a list of names |
| `glm(..., family = binomial)` | logistic regression |
| `broom::tidy()` | turns the result into a table; we keep the `dosage` row |
| `map(..., .progress = TRUE)` | runs `test_phecode()` for every phecode, with a progress bar, like a loop |

`extra_covariates` lets us add more variables to the model later (Part 6).

It takes about **5 minutes**: about 1,000 regressions on 20,000 people. A good moment to discuss the next section.

In [ ]:
test_phecode <- function(pc, extra_covariates = character()) {
  data <- phewas_samples %>%
    mutate(status = phecode_status(pc)) %>%
    filter(!is.na(status), !is.na(dosage), !is.na(PC1))

  covariates <- c("age", "sex_male", PCS, extra_covariates)
  restriction <- sex_restriction %>% filter(phecode == pc)
  if (nrow(restriction) == 1 && restriction$male_only)   data <- filter(data, sex_male == 1)
  if (nrow(restriction) == 1 && restriction$female_only) data <- filter(data, sex_male == 0)
  if (nrow(restriction) == 1 && (restriction$male_only || restriction$female_only)) {
    covariates <- setdiff(covariates, "sex_male")
  }

  n_cases <- sum(data$status == 1)
  if (n_cases < MIN_CASES) return(NULL)

  fit <- suppressWarnings(glm(reformulate(c("dosage", covariates), response = "status"),
                              family = binomial, data = data))
  broom::tidy(fit) %>%
    filter(term == "dosage") %>%
    transmute(phecode = pc, n_cases = n_cases, n_controls = sum(data$status == 0),
              OR = exp(estimate), p = p.value)
}

start <- Sys.time()
results <- map(phecodes_to_test, test_phecode, .progress = TRUE) %>% list_rbind()
Sys.time() - start

nrow(results)   # phecodes tested

**🔍 Read the output:** how long it took, and how many phecodes were tested (some drop out when exclusions leave fewer than 100 cases). In our test run: just under 5 minutes, and 992 phecodes.

## Part 6. Read the results

### The top phecodes

We add each phecode's description and group from `pheinfo`, and a **Bonferroni** threshold: 0.05 divided by the number of phecodes tested.

In [ ]:
BONFERRONI <- 0.05 / nrow(results)

results <- results %>%
  left_join(pheinfo %>% select(phecode, description, group, groupnum), by = "phecode") %>%
  mutate(group    = replace_na(group, "other"),      # a few phecodes have no group in pheinfo
         groupnum = replace_na(groupnum, 99L)) %>%
  arrange(p)

signif(BONFERRONI, 2)

results %>%
  head(15) %>%
  transmute(phecode, description, n_cases, OR = round(OR, 2), p = signif(p, 2),
            significant = p < BONFERRONI)

**🔍 Read the output:**

- **Type 2 diabetes** (250.2) and **diabetes mellitus** (250) should be at the top, with OR about **1.2**, as in 4.2 and All by All (in our test run: OR 1.20, p ≈ 10⁻⁹).
- What else passes the Bonferroni threshold? In our test run: **insulin pump user**, and type 2 diabetes **with complications** (circulatory, neurological). Just below the line: diabetic retinopathy, diabetic polyneuropathy, and gastroparesis (a stomach problem often caused by diabetes).
- The complications have **larger** ORs (about 1.3 to 1.4) than diabetes itself. **💬 Discuss:** why might that be? (Complications come with longer-lasting or harder-to-control diabetes; people with more T alleles may develop diabetes earlier or need insulin sooner.)
- **Morbid obesity** has an OR **below 1** (0.90 in our test run): more T alleles, *less* obesity. Remember the smaller waist in All by All.
- Phecodes with an OR **below 1** are linked to **fewer** T alleles.

### The PheWAS plot

The classic PheWAS plot: one point per phecode, grouped by disease **group** along the x-axis, with −log₁₀(p) up the y-axis. Triangles point **up** when the T allele goes with **higher** odds, **down** for lower. Groups alternate between two shades so you can see where each one starts. Phecodes that pass the Bonferroni line are labelled.

In [ ]:
plot_data <- results %>%
  arrange(groupnum, phecode) %>%
  mutate(x         = row_number(),
         direction = if_else(OR > 1, "Higher odds with T", "Lower odds with T"),
         shade     = factor(groupnum %% 2))

group_labels <- plot_data %>% group_by(group) %>% summarise(mid = mean(x))

ggplot(plot_data, aes(x, -log10(p))) +
  geom_point(aes(shape = direction, colour = shade), size = 2) +
  geom_hline(yintercept = -log10(BONFERRONI), linetype = "dashed") +
  geom_text(data = filter(plot_data, p < BONFERRONI), aes(label = description),
            hjust = -0.08, size = 3, check_overlap = TRUE) +
  scale_x_continuous(breaks = group_labels$mid, labels = group_labels$group) +
  scale_shape_manual(values = c("Higher odds with T" = 17, "Lower odds with T" = 25)) +
  scale_colour_manual(values = c("0" = "#3182bd", "1" = "#08306b"), guide = "none") +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 8), panel.grid.minor = element_blank()) +
  labs(x = NULL, y = "-log10(p)", shape = NULL,
       title = "PheWAS of rs7903146 (T allele)",
       subtitle = "20,000 random All of Us participants. Dashed line: Bonferroni threshold.")

**🔍 Read the output:** the highest points should be in **endocrine/metabolic** (diabetes). Compare with the All by All plot from Part 2: the same top hits, but far fewer significant ones, because we have 20,000 people instead of the hundreds of thousands in All by All.

### Direct effects, or consequences of diabetes?

If rs7903146 affects other diseases **only through** type 2 diabetes, then adjusting for diabetes should make those associations disappear. We add `has_t2d` (a case in our cohort query) as a covariate and test the strongest non-diabetes phecodes again.

In [ ]:
phewas_samples <- phewas_samples %>% mutate(has_t2d = if_else(cohort == "case", 1, 0))

top_other <- results %>%
  filter(!str_starts(phecode, "250")) %>%   # leave out the diabetes phecodes
  head(5) %>%                               # the 5 with the smallest p-values
  pull(phecode)

adjusted <- map(top_other, test_phecode, extra_covariates = "has_t2d") %>% list_rbind()

results %>%
  filter(phecode %in% top_other) %>%
  select(phecode, description, OR, p) %>%
  left_join(adjusted %>% select(phecode, OR_adj_t2d = OR, p_adj_t2d = p), by = "phecode") %>%
  mutate(across(c(OR, OR_adj_t2d), ~ round(.x, 2)), across(c(p, p_adj_t2d), ~ signif(.x, 2)))

**🔍 Read the output:** for the 5 non-diabetes phecodes with the smallest p-values, the OR and p-value **before** and **after** adjusting for type 2 diabetes. Focus on any that were **significant** before adjusting.

- If the OR moves to about **1** and p becomes large, the link was probably **through diabetes**: a consequence, not a separate effect.
- If it **stays**, rs7903146 may affect that trait in another way (**pleiotropy**), or our diabetes variable is too crude: `has_t2d` is yes/no, and says nothing about how long or how severe someone's diabetes is. Gastroparesis, for example, is a known diabetes complication, yet in our test run its OR barely moved (1.37 → 1.30).

### A warning: adjusting can also **create** an association

In our test run, **morbid obesity** went from OR **0.90** (p ≈ 0.003) to **0.85** (p ≈ 8 × 10⁻⁶) after adjusting for diabetes: *stronger*, not weaker. Why?

Both the **T allele** and **obesity** cause type 2 diabetes:

```
T allele ──→ type 2 diabetes ←── obesity
```

Diabetes is a **common effect** of both (a **collider**). Among people **with** diabetes, if someone has few T alleles, something else, often obesity, probably explains their diabetes; so within that group, T alleles and obesity become **negatively** linked, even if they weren't before. Adjusting for diabetes does exactly that (compares people *within* diabetes groups), so it can **create** or **exaggerate** a negative association. This is **collider bias**.

So which estimate should we trust? The **unadjusted** 0.90 is the cleaner one for obesity. That it's below 1 agrees with the smaller waist in All by All, and with what's known about *TCF7L2*: it raises diabetes risk through insulin secretion, not through weight. But part of the stronger 0.85 is probably bias.

**💬 Discuss:** adjusting for a variable is not always "more careful". Before adjusting, draw the arrows: is the variable a **confounder** (causes both, like ancestry in 4.2), a step on the **path** (a mediator), or a **common effect** (a collider)? Only confounders should be adjusted for, unless, as here, you're asking a specific question and know the risks.

### ✍️ Your turn

Test rs7903146 against **phecode 278.1, Obesity** (the parent of morbid obesity, 278.11). Is the T allele linked to obesity? How does that fit with the waist circumference result in All by All (Part 2)?

<details>
<summary><b>Show answer</b></summary>

```r
test_phecode("278.1")
results %>% filter(phecode == "278.1")
```

You'll probably see an OR **below 1** (in our test run, morbid obesity had 0.90). Like the smaller waist in All by All, this fits the T allele raising diabetes risk through **insulin secretion**, not through weight. If you add `extra_covariates = "has_t2d"`, the OR will probably move further below 1: that's the **collider bias** from the section above, so prefer the unadjusted result here.

</details>

In [ ]:
# Your code here

## Save the results

Summary statistics for phecodes with at least 100 cases: no individual data.

In [ ]:
save_data(results, "phewas_rs7903146.csv")

## Other PheWAS tools

| Tool | Language | Notes |
|---|---|---|
| **PheTK** | Python | built by *All of Us* researchers for this data. In our test, installing it replaced packages that **Hail** needs (`numpy`), breaking 4.3. Use it in a separate Python environment, or a separate app |
| **PheWAS** R package | R | where our lookup tables come from. It doesn't install in the standard app (a package it needs fails to build) |
| **All by All** | browser, Hail tables | ready-made results for about 3,400 phenotypes; the first place to look |

Today we did by hand what these tools automate, so you know what's inside them.

## ⚠️ Common errors

| Message | Means |
|---|---|
| *cannot open URL* … `raw.githubusercontent.com` | The phecode tables couldn't download. Check the app has internet access, and rerun the cell |
| *Query exceeds … maximum length* / *query too large* | Too many IDs in `IN (...)`. Keep `N_PEOPLE` at 20,000 or below |
| A query takes very long or fails with *Resources exceeded* | Rerun once; BigQuery is sometimes busy. If it fails again, lower `N_PEOPLE` |
| *object 'phecode_map' not found* | The `load()` cell didn't run, or a download failed |
| `geno_col` doesn't end in `_T` | `count_allele.txt` wasn't used: check the `--export-allele` part of the PLINK command |
| `glm` warnings about *fitted probabilities 0 or 1* (if you remove `suppressWarnings`) | Very common with rare outcomes; the results for that phecode are less reliable |

## ✅ Key takeaways

- A **PheWAS** tests **one variant** against **many phenotypes**, with the same regression as a GWAS
- **Phecodes** turn thousands of ICD codes into about 1,800 research diseases, with rules for cases (2+ days), exclusions, and sex
- A PheWAS needs a **random** sample: a sample chosen by disease status creates **selection bias**
- Many PheWAS hits are **consequences** of the main disease; adjust for it to look for **other** effects
- **All by All** has ready-made results from hundreds of thousands of participants: free to look up, and a check on your own work

**Next:** **4.5 rnaseq**: from DNA to **gene activity**. Is *TCF7L2* (or anything else) expressed differently in people with type 2 diabetes?

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).